# Predictive Analytics for Microarray Data

**Objective:** Analyze high-dimensional microarray-style data using feature selection and predictive classification models.

**Dataset:** Microarray gene expression dataset or synthetic high-dimensional gene-expression sample

This notebook is Colab-ready and saves tables, metrics, and visual outputs under
`results/`. Public datasets or compact sample datasets are used so the workflow
remains reproducible.


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
X, y = make_classification(
    n_samples=180,
    n_features=2000,
    n_informative=45,
    n_redundant=20,
    n_classes=2,
    random_state=42,
    class_sep=1.8,
)
gene_names = [f"gene_{i:04d}" for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=gene_names)
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.3, random_state=42, stratify=y)

models = {
    "SVM": SVC(kernel="linear", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42),
    "Logistic": LogisticRegression(max_iter=1000),
}

rows = []
fitted = {}
for name, classifier in models.items():
    pipe = Pipeline(
        [
            ("scale", StandardScaler()),
            ("select", SelectKBest(f_classif, k=50)),
            ("model", classifier),
        ]
    )
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({"model": name, "accuracy": accuracy_score(y_test, pred), "f1": f1_score(y_test, pred)})
    fitted[name] = pipe

metrics = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
metrics.to_csv(RESULTS_DIR / "microarray_model_metrics.csv", index=False)
display(metrics)


In [ ]:
best = fitted[metrics.iloc[0]["model"]]
selector = best.named_steps["select"]
selected_genes = pd.DataFrame(
    {
        "gene": X_train.columns[selector.get_support()],
        "score": selector.scores_[selector.get_support()],
    }
).sort_values("score", ascending=False)
selected_genes.to_csv(RESULTS_DIR / "selected_genes.csv", index=False)

top_genes = selected_genes.head(20)["gene"].tolist()
heatmap_data = X_df.loc[:30, top_genes]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(heatmap_data.T, cmap="coolwarm", center=0, ax=axes[0], cbar=False)
axes[0].set_title("Top Gene Expression Heatmap")
sns.barplot(data=metrics, x="model", y="accuracy", ax=axes[1])
axes[1].set_ylim(0, 1)
axes[1].set_title("Classifier Accuracy")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "microarray_dashboard.png", dpi=180)
plt.show()
